# 🖥️ Drishti-Kavach: Universal Local Hardware Training on Google Colab
### Compatible with macOS (Apple Silicon M-Series / Intel) and Windows 10/11 (NVIDIA CUDA / CPU)

Train **YOLO11-seg (RailDrishti)** using Google Colab as your browser interface while executing **100% locally on your computer's native hardware**.

### ✨ Key Advantages:
1. **🚀 ZERO GOOGLE DRIVE UPLOADS NEEDED:** Directly reads `dataset_rail-drishti/` from your local high-speed SSD.
2. **🔄 Interactive Pause & Resume:** Toggle `RESUME_TRAINING = True` at any time to resume from the last saved epoch with 0% data loss.
3. **⚡ Native Hardware Acceleration:** Automatically detects Apple Silicon Metal (MPS) on Mac or NVIDIA CUDA on Windows.
4. **📊 Minimal 3-Line Per-Epoch Metrics:** Live color-coded accuracy evaluation (Track Segm mAP, Obstacle Detection mAP, and Overall mAP).
5. **🏆 Auto-Export:** Best trained model is saved directly into `models/RailDrishti.pt` on your computer.

## 🛠️ Step 1: Connect this Notebook to your Local Computer

### 🍏 On macOS (Apple Silicon / Intel):
1. Open Terminal in your repository folder and run the launcher:
   ```bash
   cd /Users/alvinsonny/Desktop/drishti-kavach
   .venv/bin/python src/4_colab_training/launch_colab_server.py
   ```
2. Copy the URL printed in Terminal (e.g. `http://localhost:8888/?token=abcdef123456...`).
3. In this Colab Notebook, click the **downward arrow (▼)** next to **Connect** (top right) $\rightarrow$ **Connect to a local runtime** $\rightarrow$ paste the URL $\rightarrow$ Click **Connect**.

### 🪟 On Windows (NVIDIA CUDA / CPU):
1. Open PowerShell / Command Prompt in your repository folder and run:
   ```powershell
   python src/4_colab_training/launch_colab_server.py
   ```
2. Copy the `http://localhost:8888/?token=...` URL and paste into Colab $\rightarrow$ **Connect to a local runtime**.

### ⚡ Step 2: Auto-Detect Connected Local Hardware (Mac / Windows)

In [ ]:
import os, sys, platform, torch

os_name = platform.system()
print('=' * 70)
print(' 🖥️ LOCAL HARDWARE PLATFORM DETECTED')
print('=' * 70)
print(f' • Operating System : {os_name} {platform.release()} ({platform.machine()})')
print(f' • Python Executable: {sys.executable}')
print(f' • PyTorch Version  : {torch.__version__}')

# Automatic device selection
if torch.backends.mps.is_available():
    DEVICE = 'mps'
    print(' • Hardware Engine  : 🍏 Apple Silicon Metal Performance Shaders (MPS)')
elif torch.cuda.is_available():
    DEVICE = 0
    print(f' • Hardware Engine  : 🎮 NVIDIA GPU via CUDA ({torch.cuda.get_device_name(0)})')
else:
    DEVICE = 'cpu'
    print(' • Hardware Engine  : 💻 Multi-Core CPU')
print('=' * 70)

### 📊 Step 3: Setup Minimal Accuracy Monitor & Auto-Deploy Callback

In [ ]:
import os, shutil
from pathlib import Path

def get_metric_color(val: float) -> str:
    if val >= 90.0: return '\033[1;92m'
    elif val >= 80.0: return '\033[92m'
    elif val >= 65.0: return '\033[93m'
    elif val >= 45.0: return '\033[95m'
    else: return '\033[91m'

class LocalColabStatusMonitor:
    TARGET_BOX_MAP = 85.0
    TARGET_SEG_MAP = 90.0
    TARGET_OVERALL_MAP = 85.0

    def __init__(self, output_model_path: str = 'models/RailDrishti.pt'):
        self.best_map = 0.0
        self.best_epoch = 0
        self.output_model_path = output_model_path

    def on_fit_epoch_end(self, trainer):
        epoch = trainer.epoch + 1
        total_epochs = trainer.epochs
        metrics = getattr(trainer, 'metrics', {}) or {}
        box_map50 = (metrics.get('metrics/mAP50(B)', 0.0) or 0.0) * 100.0
        seg_map50 = (metrics.get('metrics/mAP50(M)', 0.0) or 0.0) * 100.0
        overall_map50 = (box_map50 + seg_map50) / 2.0 if (box_map50 > 0 and seg_map50 > 0) else (box_map50 or seg_map50 or 0.0)
        loss_val = float(trainer.tloss.mean()) if hasattr(trainer, 'tloss') and trainer.tloss is not None else None
        
        is_new_best = False
        if overall_map50 > self.best_map and overall_map50 > 1.0:
            self.best_map = overall_map50
            self.best_epoch = epoch
            is_new_best = True
            if hasattr(trainer, 'best') and os.path.exists(str(trainer.best)):
                try:
                    os.makedirs(os.path.dirname(self.output_model_path), exist_ok=True)
                    shutil.copy(str(trainer.best), self.output_model_path)
                except Exception: pass

        c_overall = get_metric_color(overall_map50)
        c_seg = get_metric_color(seg_map50)
        c_box = get_metric_color(box_map50)
        c_best = get_metric_color(self.best_map)
        rst = '\033[0m'
        loss_str = f'{loss_val:.4f}' if loss_val is not None else 'N/A'
        best_msg = f' (★ Saved to {self.output_model_path})' if is_new_best else ''

        print(f'\nEpoch [{epoch:02d}/{total_epochs:02d}] -> Current Accuracy: {c_overall}{overall_map50:.1f}%{rst} (Expected: ≥{self.TARGET_OVERALL_MAP:.0f}%) | Loss: {loss_str}')
        print(f'Metrics: Track Segm mAP: {c_seg}{seg_map50:.1f}%{rst} (Expected: ≥{self.TARGET_SEG_MAP:.0f}%) | Obstacle Box mAP: {c_box}{box_map50:.1f}%{rst} (Expected: ≥{self.TARGET_BOX_MAP:.0f}%)')
        print(f'Best Accuracy: {c_best}{self.best_map:.1f}%{rst} at Epoch {self.best_epoch}{best_msg}\n')

### 🚀 Step 4: Run Training with Pause & Resume Support
- Set `RESUME_TRAINING = False` to start a fresh training run.
- Set `RESUME_TRAINING = True` to resume from the latest saved checkpoint (`runs/.../weights/last.pt`).

In [ ]:
import glob
from ultralytics import YOLO

# ── CONFIGURATION ──────────────────────────────────────────────────────────
RESUME_TRAINING = False       # Set to True to resume from last paused checkpoint
EPOCHS = 60
BATCH_SIZE = 16              # 16 for Apple Silicon M4 / GPU, 8 for CPU
IMGSZ = 640                  # Fast local 640px resolution
BASE_MODEL = 'yolo11s-seg.pt'
DATASET_YAML = 'configs/raildrishti_dataset.yaml'
OUTPUT_MODEL = 'models/RailDrishti.pt'
# ──────────────────────────────────────────────────────────────────────────

# Find latest checkpoint if resuming
def find_local_checkpoint():
    ckpts = glob.glob('runs/**/weights/last.pt', recursive=True)
    if not ckpts: return None
    ckpts.sort(key=lambda p: os.path.getmtime(p), reverse=True)
    return ckpts[0]

monitor = LocalColabStatusMonitor(output_model_path=OUTPUT_MODEL)
last_ckpt = find_local_checkpoint()

if RESUME_TRAINING and last_ckpt and os.path.exists(last_ckpt):
    print(f'[*] Resuming training from local checkpoint: {last_ckpt}...')
    model = YOLO(last_ckpt)
    model.add_callback('on_fit_epoch_end', monitor.on_fit_epoch_end)
    model.train(resume=True)
else:
    if RESUME_TRAINING:
        print('[!] No previous checkpoint found to resume. Starting fresh instead...')
    print(f'[*] Starting fresh training with base weights: {BASE_MODEL} on device: {DEVICE}...')
    model = YOLO(BASE_MODEL)
    model.add_callback('on_fit_epoch_end', monitor.on_fit_epoch_end)
    model.train(
        data=DATASET_YAML,
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        imgsz=IMGSZ,
        device=DEVICE,
        workers=4,
        project='runs/train_local',
        name='raildrishti_local',
        exist_ok=True,
        amp=True,
        optimizer='AdamW',
        lr0=0.002,
        cos_lr=True,
        weight_decay=0.0005,
        warmup_epochs=3.0,
        patience=15,
        mosaic=1.0,
        mixup=0.1,
        fliplr=0.5,
        hsv_h=0.015,
        hsv_s=0.6,
        hsv_v=0.4
    )